# Démo : Interface de l'assistant RAG

**Contexte :** assistant du service client de Lagune Banque.

Cette démo réunit les capacités développées dans le module. Chaque option peut être activée séparément afin d'observer son effet sur la recherche et la réponse.

**Sommaire**
- [Configuration](#configuration)
- [Index et recherche](#index-recherche)
- [Génération de la réponse](#generation)
- [Interface](#interface)
- [Essais](#essais)

Les options disponibles sont la reformulation, la recherche hybride, le reranking et le contrôle des réponses. Le module fournit l'interface ; l'exercice porte sur la fonction `answer`.

| Case | Capacité | Notebook |
|---|---|---|
| Reformuler la question | query rewriting, utile pour les suites de conversation | démo 2 |
| Recherche hybride | BM25 fusionné avec les vecteurs (RRF) | TP2 |
| Reranker | reclassement des chunks par un modèle spécialisé | TP2 |
| Réponse contrôlée | citations vérifiées et refus hors corpus | TP3 |

Vous n'écrivez pas le front : `code_partage/front.py` fournit le thème et les blocs. Vous écrivez la fonction `answer`.

## <a id="configuration"></a>Configuration

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../starter/code_partage")] if (p / "rag.py").exists()).resolve()))
import rag
import rag_avance as ra
from config import prepare
from weaviate_store import connect
import time

import gradio as gr

import front

In [ ]:
prepare(embedding=True, rerank=True)

## <a id="index-recherche"></a>Index et recherche

Même index qu'aux TP : récursif, 500 caractères, recouvrement 75. `retrieve` combine les capacités selon les cases cochées : la recherche hybride ajoute BM25 aux vecteurs, le reranker garde les 3 meilleurs chunks.

In [ ]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
chunks = rag.split_documents(documents, 500, 75, "front")
index = rag.index_chunks(connect(), "demo_front", chunks)
bm25 = ra.BM25(chunks)


def rrf(listes, k=60, n=10):
    scores, vus = {}, {}
    for liste in listes:
        for rang, p in enumerate(liste, 1):
            scores[p["texte"]] = scores.get(p["texte"], 0) + 1 / (k + rang)
            vus[p["texte"]] = p
    return [vus[t] for t, _ in sorted(scores.items(), key=lambda x: -x[1])[:n]]


def retrieve(question, hybrid, rerank):
    candidates = rag.search(index, question, 20, rag.AUJOURDHUI)
    if hybrid:
        keywords = [p for p in bm25.search(question, 20) if rag.is_in_force(p["meta"], rag.AUJOURDHUI)]
        candidates = rrf([candidates, keywords], n=20)
    return rag.rerank(question, candidates, 3) if rerank else candidates[:5]

## <a id="generation"></a>Génération de la réponse

Gradio appelle `answer` à chaque message, avec la valeur des cases cochées. La fonction produit une liste de messages : des blocs repliables (question reformulée, chunks, citations vérifiées), puis la réponse. Les références `[DOC-xx]` sont retirées du texte (`front.without_references`) : les sources sont déjà dans les blocs.

La réponse contrôlée a besoin du score du reranker pour décider du refus : elle active donc le reranker.

In [ ]:
def answer(message, historique, rewrite=False, hybrid=False, rerank=False, controlled=False):
    start = time.time()
    previous = " ".join(m["content"] for m in front.exchanges(historique)[-2:])
    question = ra.rewrite(message, previous) if rewrite else message
    blocks = [front.block("Question reformulée", question)] if rewrite else []
    found = retrieve(question, hybrid, rerank or controlled)
    blocks.append(front.chunks_block(found, round(time.time() - start, 1)))
    yield blocks                                          # les blocs s'affichent pendant que le modèle rédige
    if controlled:
        result = ra.answer_with_sources(question, found)
        if result["citations"]:
            blocks.append(front.block("Citations vérifiées", "\n\n".join(f"**{c['source']}** : « {c['extrait']} »" for c in result["citations"])))
        text = result["reponse"]
    else:
        text = rag.generate(rag.build_prompt(question, found))
    yield blocks + [gr.ChatMessage(content=front.without_references(text))]

Test de la fonction sans l'interface, avec toutes les capacités puis aucune :

In [ ]:
for options in [(True, True, True, True), (False, False, False, False)]:
    for sortie in answer("Quels sont les frais de retard du Prêt Express ?", [], *options):
        pass
    print(options, "->", sortie[-1].content)

## <a id="interface"></a>Interface

`front.create` fabrique le chat à partir de la fonction ; `champs` ajoute les cases à cocher, et chaque exemple donne la question puis la valeur de chaque case. `front.launch` affiche l'interface sous la cellule.

In [ ]:
cases = [gr.Checkbox(label="Reformuler la question", value=False), gr.Checkbox(label="Recherche hybride (BM25)", value=False),
         gr.Checkbox(label="Reranker", value=False), gr.Checkbox(label="Réponse contrôlée (citations vérifiées, refus)", value=False)]
demo = front.create(
    answer,
    titre="Assistant Lagune Banque",
    description="Posez vos questions sur les produits, les frais et les procédures de Lagune Banque (données fictives). Les chunks utilisés s'affichent sous chaque réponse.",
    accueil="**Bonjour, comment puis-je vous aider ?**<br>Cochez les capacités à activer, puis posez une question.",
    exemples=[["Quels sont les frais de retard du Prêt Express ?", False, False, False, False],
              ["Quels sont les frais de retard du Prêt Express ?", False, False, True, True],
              ["Et pour le Prêt Express+ ?", True, False, True, True],
              ["Quelle est la météo prévue à Abidjan demain ?", False, False, False, True]],
    champs=cases, titre_champs="Capacités de l'assistant")
front.launch(demo)

## <a id="essais"></a>Essais

1. `Quels sont les frais de retard du Prêt Express ?` sans aucune case, puis avec **Reranker** : les chunks changent d'ordre, et le bloc affiche un score au lieu d'une distance.
2. `Quelle est la météo prévue à Abidjan demain ?` sans case, puis avec **Réponse contrôlée** : le modèle répond librement dans le premier cas, l'assistant refuse dans le second.
3. Posez `Quels sont les frais de retard du Prêt Express ?` puis `Et pour le Prêt Express+ ?` avec **Reformuler la question** : le message court est complété avant la recherche.
4. Cochez tout et observez le temps affiché dans le bloc des chunks : chaque capacité a un coût.

Pour ouvrir l'interface dans un onglet du navigateur : `front.launch(demo, navigateur=True)`.

In [ ]:
demo.close()